In [ ]:
# ============================================================
# IMDB Movie Review Sentiment Analysis
# File: src/train_model.py
# ============================================================

import os
import re
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ============================================================
# Create folders
# ============================================================

os.makedirs("models", exist_ok=True)

# ============================================================
# Load Dataset
# ============================================================

DATASET_PATH = r"C:\Users\Dell\Downloads\archive\IMDB Dataset\IMDB Dataset.csv"

data = pd.read_csv(DATASET_PATH )

print(data.head())

print("\nDataset Shape:")
print(data.shape)

print("\nMissing Values:")
print(data.isnull().sum())

# ============================================================
# Remove Missing Values
# ============================================================

data.dropna(inplace=True)

# ============================================================
# Text Cleaning
# ============================================================

def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z ]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

data["review"] = data["review"].apply(clean_text)

# ============================================================
# Encode Labels
# ============================================================

encoder = LabelEncoder()

data["sentiment"] = encoder.fit_transform(data["sentiment"])

joblib.dump(encoder, "models/label_encoder.pkl")

# ============================================================
# Features & Labels
# ============================================================

X = data["review"]

y = data["sentiment"]

# ============================================================
# Tokenization
# ============================================================

VOCAB_SIZE = 10000

tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X)

joblib.dump(
    tokenizer,
    "models/tokenizer.pkl"
)

# ============================================================
# Convert Text to Sequences
# ============================================================

sequences = tokenizer.texts_to_sequences(X)

MAX_LENGTH = 200

X = pad_sequences(
    sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

# ============================================================
# Train Test Split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining Samples:", len(X_train))
print("Testing Samples :", len(X_test))

# ============================================================
# Build LSTM Model
# ============================================================

model = Sequential()

model.add(
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128,
        input_length=MAX_LENGTH
    )
)

model.add(
    LSTM(
        128,
        dropout=0.2,
        recurrent_dropout=0.2
    )
)

model.add(
    Dropout(0.5)
)

model.add(
    Dense(
        64,
        activation="relu"
    )
)

model.add(
    Dense(
        1,
        activation="sigmoid"
    )
)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# ============================================================
# Train Model
# ============================================================

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=5,
    batch_size=64,
    verbose=1
)

# ============================================================
# Evaluate
# ============================================================

loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("\nTest Accuracy:", accuracy)

# ============================================================
# Save Model
# ============================================================

model.save("models/sentiment_model.keras")

print("\nModel Saved Successfully!")
print("Tokenizer Saved Successfully!")
print("Label Encoder Saved Successfully!")

# ============================================================
# IMDB Movie Review Sentiment Analysis
# File: src/predict.py
# ============================================================

import re
import joblib
import numpy as np

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ============================================================
# Configuration
# ============================================================

MODEL_PATH = "models/sentiment_model.keras"
TOKENIZER_PATH = "models/tokenizer.pkl"
LABEL_ENCODER_PATH = "models/label_encoder.pkl"

MAX_LENGTH = 200

# ============================================================
# Load Saved Files
# ============================================================

print("Loading Model...")

model = load_model(MODEL_PATH)

print("Loading Tokenizer...")

tokenizer = joblib.load(TOKENIZER_PATH)

print("Loading Label Encoder...")

label_encoder = joblib.load(LABEL_ENCODER_PATH)

print("Everything Loaded Successfully!")

# ============================================================
# Text Cleaning Function
# ============================================================

def clean_text(text):

    text = text.lower()

    text = re.sub(r"<.*?>", "", text)

    text = re.sub(r"[^a-zA-Z ]", "", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

# ============================================================
# Predict Function
# ============================================================

def predict_sentiment(review):

    review = clean_text(review)

    sequence = tokenizer.texts_to_sequences([review])

    padded = pad_sequences(
        sequence,
        maxlen=MAX_LENGTH,
        padding="post",
        truncating="post"
    )

    prediction = model.predict(padded, verbose=0)[0][0]

    if prediction >= 0.5:
        sentiment = "Positive 😊"
        confidence = prediction
    else:
        sentiment = "Negative 😞"
        confidence = 1 - prediction

    return sentiment, float(confidence)

# ============================================================
# Test Prediction
# ============================================================

if __name__ == "__main__":

    while True:

        review = input("\nEnter Movie Review (type 'exit' to quit): ")

        if review.lower() == "exit":
            break

        sentiment, confidence = predict_sentiment(review)

        print("\nPrediction :", sentiment)
        print("Confidence :", round(confidence * 100, 2), "%")

        # ============================================================
# IMDB Movie Review Sentiment Analysis
# File: app.py
# ============================================================

import re
import joblib
import gradio as gr

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ============================================================
# Configuration
# ============================================================

MODEL_PATH = "models/sentiment_model.keras"
TOKENIZER_PATH = "models/tokenizer.pkl"
LABEL_ENCODER_PATH = "models/label_encoder.pkl"

MAX_LENGTH = 200

# ============================================================
# Load Model
# ============================================================

print("Loading Model...")

model = load_model(MODEL_PATH)

tokenizer = joblib.load(TOKENIZER_PATH)

label_encoder = joblib.load(LABEL_ENCODER_PATH)

print("Application Ready!")

# ============================================================
# Text Cleaning
# ============================================================

def clean_text(text):

    text = text.lower()

    text = re.sub(r"<.*?>", "", text)

    text = re.sub(r"[^a-zA-Z ]", "", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

# ============================================================
# Prediction Function
# ============================================================

def predict_review(review):

    if review.strip() == "":
        return "⚠ Please enter a movie review.", ""

    review = clean_text(review)

    sequence = tokenizer.texts_to_sequences([review])

    padded = pad_sequences(
        sequence,
        maxlen=MAX_LENGTH,
        padding="post",
        truncating="post"
    )

    probability = model.predict(padded, verbose=0)[0][0]

    if probability >= 0.5:
        sentiment = "😊 Positive Review"
        confidence = probability * 100
    else:
        sentiment = "😞 Negative Review"
        confidence = (1 - probability) * 100

    result = f"{sentiment}\n\nConfidence : {confidence:.2f}%"

    return result, confidence

# ============================================================
# Examples
# ============================================================

examples = [
    ["This movie is fantastic. I loved every minute."],
    ["One of the best movies ever made."],
    ["Worst movie I have ever watched."],
    ["The acting was terrible and boring."],
    ["Excellent story and amazing acting."]
]

# ============================================================
# Gradio Interface
# ============================================================

app = gr.Interface(
    fn=predict_review,

    inputs=gr.Textbox(
        lines=8,
        placeholder="Enter your movie review here...",
        label="Movie Review"
    ),

    outputs=[
        gr.Textbox(label="Prediction"),
        gr.Number(label="Confidence (%)")
    ],

    title="🎬 IMDB Movie Review Sentiment Analysis",

    description="""
Enter any movie review and the AI model will predict whether
the review is Positive or Negative.
""",

    examples=examples,

    theme="soft"
)

# ============================================================
# Launch
# ============================================================

if __name__ == "__main__":

    app.launch(
        debug=True,
        share=True
 )


                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Dataset Shape:
(50000, 2)

Missing Values:
review       0
sentiment    0
dtype: int64

Training Samples: 40000
Testing Samples : 10000


C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 104s 163ms/step - accuracy: 0.5303 - loss: 0.6896 - val_accuracy: 0.6102 - val_loss: 0.6600
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 127s 202ms/step - accuracy: 0.6059 - loss: 0.6529 - val_accuracy: 0.5478 - val_loss: 0.6771
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 127s 203ms/step - accuracy: 0.6026 - loss: 0.6385 - val_accuracy: 0.5345 - val_loss: 0.6812
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 391s 626ms/step - accuracy: 0.7501 - loss: 0.4942 - val_accuracy: 0.8533 - val_loss: 0.3604
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 111s 178ms/step - accuracy: 0.8853 - loss: 0.3006 - val_accuracy: 0.8682 - val_loss: 0.3145

Test Accuracy: 0.8682000041007996

Model Saved Successfully!
Tokenizer Saved Successfully!
Label Encoder Saved Successfully!
Loading Model...
Loading Tokenizer...
Loading Label Encoder...
Everything Loaded Successfully!

Prediction : Negative 😞
Confidence : 93.7 %

Prediction : Negative 😞
Confidence : 50.03 %
